In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

df = pd.read_parquet("../data/interactions.parquet")

print(df.shape)

df.head()

(1689188, 4)


,reviewerID,asin,overall,unixReviewTime
0,AO94DHGC771SJ,0528881469,5,1370131200
1,AMO214LNFCEI4,0528881469,1,1290643200
2,A3N7T0DY83Y4IG,0528881469,3,1283990400
3,A1H8PY3QHMQQA0,0528881469,2,1290556800
4,A24EV6RXELQZ63,0528881469,1,1317254400


In [ ]:
df["date"] = pd.to_datetime(
    df["unixReviewTime"],
    unit="s"
)

df.head()

,reviewerID,asin,overall,unixReviewTime,date
0,AO94DHGC771SJ,0528881469,5,1370131200,2013-06-02
1,AMO214LNFCEI4,0528881469,1,1290643200,2010-11-25
2,A3N7T0DY83Y4IG,0528881469,3,1283990400,2010-09-09
3,A1H8PY3QHMQQA0,0528881469,2,1290556800,2010-11-24
4,A24EV6RXELQZ63,0528881469,1,1317254400,2011-09-29


In [ ]:
product_stats = (
    df.groupby("asin")
      .agg(
          avg_rating=("overall", "mean"),
          num_reviews=("overall", "count")
      )
      .reset_index()
)

product_stats.head()

,asin,avg_rating,num_reviews
0,0528881469,2.400000,5
1,0594451647,4.200000,5
2,0594481813,4.000000,8
3,0972683275,4.461187,219
4,1400501466,3.953488,43


In [4]:
C = product_stats["avg_rating"].mean()

print("Global Mean Rating:", C)

Global Mean Rating: 4.137459798367161


In [5]:
m = product_stats["num_reviews"].quantile(0.90)

print("90th Percentile Review Count:", m)

90th Percentile Review Count: 51.0


In [ ]:
product_stats["weighted_rating"] = (
    (
        product_stats["num_reviews"]
        /
        (product_stats["num_reviews"] + m)
    )
    * product_stats["avg_rating"]
)
+ (
    (
        m
        /
        (product_stats["num_reviews"] + m)
    )
    * C
)

0        3.768044
1        3.768044
2        3.576448
3        0.781520
4        2.244792
           ...   
62996    3.014435
62997    3.103095
62998    2.971978
62999    2.930701
63000    3.768044
Name: num_reviews, Length: 63001, dtype: float64

In [ ]:
top_products = (
    product_stats
    .sort_values(
        "weighted_rating",
        ascending=False
    )
)

top_products.head(20)

,asin,avg_rating,num_reviews,weighted_rating
29247,B003ES5ZUU,4.800386,4143,4.742012
16776,B0019EHU8G,4.801164,3435,4.730924
39993,B0052SCU8U,4.859760,1419,4.691156
12361,B000QUUFRW,4.793651,1890,4.667697
10739,B000LRMS66,4.745408,1960,4.625062
20979,B001TH7GUU,4.829289,1154,4.624896
25796,B002V88HFE,4.736311,2082,4.623066
60159,B00E3W15P0,4.792913,1270,4.607873
26573,B00316263Y,4.776276,1332,4.600145
29448,B003FVVMS0,4.877081,781,4.578125


In [8]:
POPULAR_PRODUCTS = (
    top_products["asin"]
    .tolist()
)

In [9]:
def popularity_recommender(k=10):
    return POPULAR_PRODUCTS[:k]

popularity_recommender()

['B003ES5ZUU',
 'B0019EHU8G',
 'B0052SCU8U',
 'B000QUUFRW',
 'B000LRMS66',
 'B001TH7GUU',
 'B002V88HFE',
 'B00E3W15P0',
 'B00316263Y',
 'B003FVVMS0']

*Leave-One-Out Train/Test Split*

In [10]:
df = df.sort_values(
    ["reviewerID", "unixReviewTime"]
)

In [11]:
test = (
    df.groupby("reviewerID")
      .tail(1)
)

In [12]:
test_idx = test.index

train = df.drop(test_idx)

In [13]:
print("Train:", train.shape)
print("Test :", test.shape)

Train: (1496785, 5)
Test : (192403, 5)


In [14]:
ground_truth = (
    test.groupby("reviewerID")["asin"]
        .apply(set)
        .to_dict()
)

list(ground_truth.items())[:3]

[('A000715434M800HLCENK9', {'B00HMZG3YS'}),
 ('A00101847G3FJTWYGNQA', {'B00B19L8LO'}),
 ('A00166281YWM98A3SVD55', {'B007B5S8BU'})]

In [15]:
def precision_at_k(
    recommended,
    relevant,
    k=10
):
    recommended = recommended[:k]

    hits = len(
        set(recommended)
        &
        set(relevant)
    )

    return hits / k

In [16]:
precisions = []

for user, relevant_items in ground_truth.items():

    recommendations = popularity_recommender(10)

    p = precision_at_k(
        recommendations,
        relevant_items,
        k=10
    )

    precisions.append(p)

In [17]:
mean_precision = np.mean(
    precisions
)

print(
    f"Popularity Precision@10: "
    f"{mean_precision:.4f}"
)

Popularity Precision@10: 0.0010


In [18]:
recommended_items = set(
    popularity_recommender(10)
)

all_items = set(
    train["asin"].unique()
)

coverage = (
    len(recommended_items)
    /
    len(all_items)
)

print(
    f"Coverage: {coverage:.6f}"
)

Coverage: 0.000159


In [19]:
product_stats.to_parquet(
    "../data/product_stats.parquet",
    index=False
)

In [20]:
train["interaction"] = (
    train["overall"] >= 4
).astype(np.int8)

train_cf = train[
    train["interaction"] == 1
].copy()

print(train_cf.shape)

train_cf["interaction"].value_counts()

(1203320, 6)


interaction
1    1203320
Name: count, dtype: int64

In [21]:
train_cf.to_parquet(
    "../data/train_cf.parquet",
    index=False
)

print("Saved train_cf.parquet")

Saved train_cf.parquet


In [22]:
print("Unique users:", train_cf["reviewerID"].nunique())
print("Unique items:", train_cf["asin"].nunique())
print("Interactions:", len(train_cf))

Unique users: 190963
Unique items: 62707
Interactions: 1203320


In [23]:
train.to_parquet(
    "../data/train.parquet",
    index=False
)

test.to_parquet(
    "../data/test.parquet",
    index=False
)